# Predict telco customer churn with Kumo Engine

Which mobile and broadband customers will cancel in the next 30 days?

Churn signals are spread across many tables: contracts, weekly usage, bills and support tickets. Kumo Engine trains a graph model directly on those related tables, so you do not hand-build features such as "dropped calls in the last 30 days" or "months until contract end". It reads each customer's history up to every prediction time, so training never sees the future. Use it for a churn model trained on your own data and rescored as new data lands. For a pretrained baseline on a single feature table, see the [KumoTabular churn notebook](../../../structured-data-models/tabular/telco/notebooks/local_structured_data_models_telco_churn.ipynb).

The workflow: describe the tables as a graph, state the prediction as a predictive query, train, then score the active base after a new month of data arrives.

Use a Linux notebook kernel with a supported GPU and local SSD scratch. Complete [Linux deployment setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/linux-machine-deployment) and check [GPU sizing](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/gpu-sizing). This example follows the [binary classification workflow](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/code-examples/linux-deployment/binary-classification). All data is synthetic, so metrics are **demonstration only**.

## Install

Install the Engine SDK and the data libraries used by this notebook.

In [1]:
%pip install -q kumo-engine pandas numpy pyarrow

## Import the local Engine APIs

In [2]:
import os
import shutil
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
import pandas as pd
from IPython.display import display

import kumo.engine as kumo
from kumo.engine.trainer.output_config import ModelOutputConfig, PredictionOutputConfig, TrainingTableOutputConfig
from kumoapi.graph import Edge
from kumoapi.jobs import MetadataField, PredictionArtifactType
from kumoapi.model_plan import RunMode, TrainingTableGenerationPlan
from kumoapi.typing import Dtype, Stype

## Set the example parameters

Set `KUMO_SCRATCH` to a writable local SSD directory. The model trains on data before `TRAIN_END`; the month from `TRAIN_END` to `AS_OF` arrives later and is used only for batch scoring.

In [3]:
SCRATCH_PARENT = Path(os.environ["KUMO_SCRATCH"]).expanduser().resolve()
OUTPUT_PARENT = Path.cwd() / "kumo_example_runs"
RANDOM_SEED = 42
N_CUSTOMERS = 4000
SIM_START = pd.Timestamp("2024-01-01")
TRAIN_END = pd.Timestamp("2025-06-01")
AS_OF = pd.Timestamp("2025-07-01")
DECISION_THRESHOLD = 0.5

OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
run_dir = Path(mkdtemp(prefix="telco_churn-", dir=OUTPUT_PARENT))
scratch_dir = Path(mkdtemp(prefix="telco_churn-", dir=SCRATCH_PARENT))
source_dir, output_dir = run_dir / "source", run_dir / "output"
source_dir.mkdir()
output_dir.mkdir()
kumo.configure(scratch_dir=scratch_dir)
destination = kumo.FileConnector(root_dir=str(output_dir))
print("Run:", run_dir)

Run: /home/ubuntu/telco-churn/nb/kumo_example_runs/telco_churn-934wo6cn


## Generate synthetic example data

Run the next two cells to simulate 18 months of a consumer telco base. Churn risk rises with network problems, bill shock when a contract ends, the April price rise for value-segment customers, late payments and declining usage; customers with two products churn less. Rows on or after `TRAIN_END` are held back as the "new month" for scoring.
You can skim the generation code; the Kumo workflow starts at **Build the graph**.

In [4]:
rng = np.random.default_rng(RANDOM_SEED)
REGIONS = ["north", "south", "east", "west", "central"]
region_dropped_calls = {"north": 0.9, "south": 0.4, "east": 0.5, "west": 0.7, "central": 0.35}
plans = pd.DataFrame({
    "plan_id": np.arange(1, 7, dtype=np.int64),
    "product": ["sim_only", "sim_only", "handset", "broadband", "broadband", "broadband_tv"],
    "contract_months": [1, 12, 24, 18, 24, 24],
    "monthly_price": [10.0, 14.0, 38.0, 30.0, 45.0, 70.0],
})


def simulate_telco(n_customers, start, end, signup_from, recent_share):
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    ids = np.arange(1, n_customers + 1, dtype=np.int64)
    recent = rng.random(n_customers) < recent_share
    signup = np.where(recent,
        start + pd.to_timedelta(rng.integers(0, (end - start).days - 30, n_customers), unit="D"),
        pd.Timestamp(signup_from) + pd.to_timedelta(rng.integers(0, (start - pd.Timestamp(signup_from)).days, n_customers), unit="D"))
    customers = pd.DataFrame({"customer_id": ids, "signup_date": pd.to_datetime(signup),
        "region": rng.choice(REGIONS, n_customers),
        "age_band": rng.choice(["18-24", "25-34", "35-54", "55+"], n_customers),
        "acquisition_channel": rng.choice(["online", "retail_store", "telesales"], n_customers),
        "segment": rng.choice(["value", "standard", "premium"], n_customers, p=[0.35, 0.45, 0.2])})

    plan_id = rng.choice(plans.plan_id, n_customers, p=[0.15, 0.2, 0.25, 0.15, 0.1, 0.15])
    contract_days = plans.set_index("plan_id").contract_months[plan_id].to_numpy() * 30
    contract_start = np.where(recent, customers.signup_date,
        start - pd.to_timedelta(rng.integers(0, contract_days), unit="D"))
    contract_start = pd.to_datetime(contract_start)
    contract_end = (contract_start + pd.to_timedelta(contract_days, unit="D")).to_numpy()
    second_line = rng.random(n_customers) < 0.3
    subscriptions = pd.DataFrame({"customer_id": np.r_[ids, ids[second_line]],
        "plan_id": np.r_[plan_id, np.where(plan_id[second_line] <= 3, 4, 2)],
        "started_at": np.r_[contract_start, contract_start[second_line]],
        "contract_end_date": np.r_[contract_end, contract_end[second_line]]})
    subscriptions.insert(0, "subscription_id", np.arange(1, len(subscriptions) + 1, dtype=np.int64))
    monthly_price = subscriptions.merge(plans, on="plan_id").groupby("customer_id").monthly_price.sum()[ids].to_numpy()

    base_usage = rng.gamma(2.0, 2.5, n_customers)
    price_sensitive = (customers.segment.to_numpy() == "value") & (rng.random(n_customers) < 0.8)
    financial_stress = rng.beta(1.5, 10, n_customers)
    dropped_rate = customers.region.map(region_dropped_calls).to_numpy()
    rolling = plan_id == 1
    dissatisfaction = np.zeros(n_customers)
    signup_dates = customers.signup_date.to_numpy()
    churned_at = np.full(n_customers, np.datetime64("NaT", "ns"))
    day = np.timedelta64(1, "D")
    last_late = np.full(n_customers, -99)
    usage, bills, tickets = [], [], []

    for week, t in enumerate(pd.date_range(start, end - np.timedelta64(7, "D"), freq="7D").to_numpy()):
        active = (signup_dates <= t) & np.isnat(churned_at)
        idx = np.flatnonzero(active)
        outage = {r: 5.0 if rng.random() < 0.04 else 1.0 for r in REGIONS}
        dropped = rng.poisson(dropped_rate[idx] * customers.region.iloc[idx].map(outage).to_numpy())
        out_of_contract = contract_end[idx] < t
        price_rise = 1.064 ** ((t >= pd.Timestamp("2024-04-01")) + (t >= pd.Timestamp("2025-04-01")))
        recent_rise = (t >= pd.Timestamp("2025-04-01")) & (t < pd.Timestamp("2025-05-15"))
        bill_shock = (out_of_contract & (contract_end[idx] > t - 60 * day)) | (recent_rise & price_sensitive[idx])

        network_ticket = rng.random(len(idx)) < 0.01 + 0.06 * dropped
        billing_ticket = rng.random(len(idx)) < 0.01 + 0.12 * bill_shock
        dissatisfaction[idx] = np.maximum(0, 0.8 * dissatisfaction[idx] + 1.2 * network_ticket
            + 1.0 * billing_ticket + 0.05 * dropped + rng.normal(0, 0.05, len(idx)))
        engagement = np.exp(-0.5 * dissatisfaction[idx])
        usage.append(pd.DataFrame({"customer_id": ids[idx], "week_start": t,
            "data_gb": (base_usage[idx] * engagement * rng.lognormal(0, 0.25, len(idx))).round(2),
            "voice_minutes": (60 * engagement * rng.lognormal(0, 0.4, len(idx))).round(0),
            "dropped_calls": dropped}))
        for flag, category in ((network_ticket, "network"), (billing_ticket, "billing")):
            tickets.append(pd.DataFrame({"customer_id": ids[idx[flag]], "category": category,
                "opened_at": t + rng.integers(0, 7, flag.sum()) * day}))
        if week % 4 == 0:
            paid_late = rng.random(len(idx)) < 0.02 + 0.6 * financial_stress[idx]
            last_late[idx[paid_late]] = week
            amount = monthly_price[idx] * price_rise * np.where(out_of_contract & ~rolling[idx], 1.3, 1.0)
            bills.append(pd.DataFrame({"customer_id": ids[idx], "billed_at": t,
                "amount": amount.round(2), "paid_late": paid_late.astype(np.int64)}))

        logit = (-8.6 + 1.6 * dissatisfaction[idx] + 2.2 * out_of_contract + 1.2 * rolling[idx]
                 + 3.0 * (recent_rise & price_sensitive[idx]) + 2.0 * (week - last_late[idx] <= 4)
                 - 1.5 * np.isin(ids[idx], subscriptions.customer_id[subscriptions.customer_id.duplicated()]))
        leaves = rng.random(len(idx)) < 1 / (1 + np.exp(-logit))
        churned_at[idx[leaves]] = t + rng.integers(0, 7, leaves.sum()) * day

    usage, bills, tickets = (pd.concat(frames, ignore_index=True) for frames in (usage, bills, tickets))
    cancellations = pd.DataFrame({"customer_id": ids, "cancelled_at": churned_at}).dropna()
    for frame, key in ((usage, "usage_id"), (bills, "bill_id"), (tickets, "ticket_id"), (cancellations, "cancellation_id")):
        frame.insert(0, key, np.arange(1, len(frame) + 1, dtype=np.int64))
    return {"customers": customers, "plans": plans, "subscriptions": subscriptions, "usage": usage,
            "bills": bills, "support_tickets": tickets, "cancellations": cancellations}

In [5]:
tables = simulate_telco(N_CUSTOMERS, SIM_START, AS_OF, signup_from="2019-01-01", recent_share=0.3)
time_columns = {"customers": "signup_date", "subscriptions": "started_at", "usage": "week_start",
                "bills": "billed_at", "support_tickets": "opened_at", "cancellations": "cancelled_at"}
new_rows = {}
for name, frame in tables.items():
    (source_dir / name).mkdir()
    is_new = frame[time_columns[name]] >= TRAIN_END if name in time_columns else np.zeros(len(frame), bool)
    frame[~is_new].to_parquet(source_dir / name / "history.parquet", index=False)
    new_rows[name] = frame[is_new]
    print(f"{name:16s} history={(~is_new).sum():7d}  new={is_new.sum():6d}")
display(tables["usage"].head(3))

customers        history=   4000  new=     0
plans            history=      6  new=     0
subscriptions    history=   5194  new=     0
usage            history= 208877  new= 10481
bills            history=  53589  new=  2616
support_tickets  history=  15732  new=   780
cancellations    history=   1350  new=    69


,usage_id,customer_id,week_start,data_gb,voice_minutes,dropped_calls
0,1,1,2024-01-01,10.27,46.0,1
1,2,2,2024-01-01,4.59,69.0,2
2,3,3,2024-01-01,7.22,91.0,0


## Build the graph

Each table becomes a graph node type and each foreign key an edge. Event tables carry a time column, so Kumo only uses history before each prediction time. See [tables and relationships](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs).

In [6]:
files = kumo.FileConnector(root_dir=str(source_dir))


def table(name, primary_key, time_column=None, categorical=(), foreign_keys=(), dates=()):
    t = kumo.Table.from_source_table(files[name], primary_key=primary_key, time_column=time_column)
    for column in ([time_column] if time_column else []) + list(dates):
        t = t.with_column(column, dtype=Dtype.date)
    for column in categorical:
        t = t.with_column(column, stype=Stype.categorical)
    for column in foreign_keys:
        t = t.with_column(column, stype=Stype.ID)
    return t


graph = kumo.Graph(
    tables={
        "customers": table("customers", "customer_id", "signup_date", ["region", "age_band", "acquisition_channel", "segment"]),
        "plans": table("plans", "plan_id", categorical=["product"]),
        "subscriptions": table("subscriptions", "subscription_id", "started_at", foreign_keys=["customer_id", "plan_id"],
                               dates=["contract_end_date"]),
        "usage": table("usage", "usage_id", "week_start", foreign_keys=["customer_id"]),
        "bills": table("bills", "bill_id", "billed_at", foreign_keys=["customer_id"]),
        "support_tickets": table("support_tickets", "ticket_id", "opened_at", ["category"], ["customer_id"]),
        "cancellations": table("cancellations", "cancellation_id", "cancelled_at", foreign_keys=["customer_id"]),
    },
    edges=[Edge(src, "customer_id", "customers") for src in ("subscriptions", "usage", "bills", "support_tickets", "cancellations")]
          + [Edge("subscriptions", "plan_id", "plans")],
)
graph.validate()

## Inspect the graph

[Graph visualization](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs) is a commented preview because the published package does not expose it yet; the relationship table below comes from the actual graph.

In [7]:
# Preview only: graph.visualize(show_cols=True)
display(pd.DataFrame([{"source": e.src_table, "foreign_key": e.fkey, "destination": e.dst_table} for e in graph.edges]))

,source,foreign_key,destination
0,subscriptions,customer_id,customers
1,usage,customer_id,customers
2,bills,customer_id,customers
3,support_tickets,customer_id,customers
4,cancellations,customer_id,customers
5,subscriptions,plan_id,plans


## Materialize the graph

Snapshot the historical source files for training.

In [8]:
snapshot = graph.snapshot()
print("Training snapshot:", snapshot.id)

Training snapshot: c977bde5a24d94ca002fb95e37c8d85ad68278328a2115feedf8963f94d7468c@c2f04935bc3dd407


## Define the prediction task

Churn is a cancellation in the 30 days after each prediction time. The `WHERE` clause restricts predictions to active customers, meaning those with usage in the previous 14 days, so customers who have already left are not scored. See [predictive queries](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries).

In [9]:
QUERY = ("PREDICT COUNT(cancellations.*, 0, 30, days) > 0 "
         "FOR EACH customers.customer_id WHERE COUNT(usage.*, -14, 0, days) > 0")
query = kumo.PredictiveQuery(graph, QUERY)
query.validate()
print("Task:", query.get_task_type())

Task: binary_classification


## Generate training rows and inspect the splits

Create monthly prediction times over the last year of history and split them chronologically (70/15/15), so validation and test are later than training, as they would be in production. In the [training table](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training-tables), `SPLIT` 0 is training, 1 validation and 2 test; `TARGET1` is the churn label.

In [10]:
generation_plan = TrainingTableGenerationPlan(
    split="ApproxDateOffsetSplit([0.7, 0.15, 0.15])",
    train_start_offset=360, train_end_offset=0, timeframe_step=30,
)
training_table = query.generate_training_table(snapshot, plan=generation_plan)
training_table.export(TrainingTableOutputConfig(connector=destination, table_name="training_rows/"))
training_rows = pd.read_parquet(output_dir / "training_rows")
display(training_rows.groupby("SPLIT").agg(rows=("ENTITY1", "size"), churn_rate=("TARGET1", "mean"),
                                            first_anchor=("TIMESTAMP", "min"), last_anchor=("TIMESTAMP", "max")))

,rows,churn_rate,first_anchor,last_anchor
SPLIT,,,,
0,25695,0.023468,2024-06-05,2025-01-31
1,5780,0.048270,2025-03-02,2025-04-01
2,2810,0.054448,2025-05-01,2025-05-01


## Configure the model

Start from the [FAST plan](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training). Churn is a minority outcome, so select the model on AUPRC and also report AUROC. Disable refitting to keep the test split unseen, and enable training explanations.

In [11]:
plan = query.suggest_model_plan(snapshot, run_mode=RunMode.FAST)
plan.training_job.refit_trainval = False
plan.training_job.refit_full = False
plan.training_job.disable_explain = False
plan.training_job.metrics = ["auprc", "auroc"]
plan.training_job.tune_metric = "auprc"
print("Tuning metric:", plan.training_job.tune_metric, "| Max epochs:", plan.optimization.max_epochs)

Tuning metric: auprc | Max epochs: 8


## Train and retain the model

Fit on the training rows and export the model bundle for reuse.

In [12]:
trainer = kumo.Trainer(plan)
fit_result = trainer.fit(training_table, snapshot=snapshot,
                         export_config=ModelOutputConfig(connector=destination, path="model"))

## Evaluate

**Synthetic-data metrics: demonstration only.** AUROC measures how well the model ranks customers who cancel within 30 days above those who stay. AUPRC is more informative when churners are rare: a random ranking scores about the test split's churn rate shown above (5.4%), so a higher AUPRC means the model surfaces churners better than random.

In [13]:
metrics = pd.Series(fit_result.metrics)[["val_auroc", "val_auprc", "test_auroc", "test_auprc"]]
display(metrics.round(3).rename_axis("metric").to_frame("value"))

,value
metric,
val_auroc,0.737
val_auprc,0.131
test_auroc,0.791
test_auprc,0.197


## Inspect training explanations

[Training explanations](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/model-explainability) rank the table columns the model is most sensitive to on test examples. Here contract end dates and tenure rank highest: contract expiry is the moment to target with retention offers. Sensitivity is not a causal effect.

In [14]:
scores_csv = next(Path(fit_result.bundle_dir).parent.rglob("explain_test/scores.csv"))
sensitivity = pd.read_csv(scores_csv).sort_values("score", ascending=False)
display(sensitivity[["table", "column", "score"]].head(8))

,table,column,score
0,subscriptions,contract_end_date,0.352105
1,subscriptions,started_at,0.282906
2,customers,signup_date,0.202447
3,support_tickets,opened_at,0.088562
4,customers,region,0.014972
5,customers,acquisition_channel,0.014872
6,customers,segment,0.011432
7,customers,age_band,0.010234


## Batch predictions

A new month of data arrives. Append the usage, bills, support tickets and cancellations recorded from `TRAIN_END` to `AS_OF` as new Parquet parts, then snapshot again. The training snapshot is unchanged.

In [15]:
for name, frame in new_rows.items():
    if len(frame):
        frame.to_parquet(source_dir / name / "new.parquet", index=False)
scoring_snapshot = graph.snapshot()
print("Scoring snapshot:", scoring_snapshot.id)

Scoring snapshot: c977bde5a24d94ca002fb95e37c8d85ad68278328a2115feedf8963f94d7468c@3592239bcd4aa59c


### Prepare the scoring population

Score every customer active at `AS_OF`: signed up, not cancelled and with usage in the last 14 days. A [custom prediction table](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/making-predictions) lists those customer IDs with the scoring time and no labels.

In [16]:
recent_usage = tables["usage"].loc[tables["usage"].week_start >= AS_OF - np.timedelta64(14, "D"), "customer_id"]
active_ids = np.setdiff1d(recent_usage.unique(), tables["cancellations"].customer_id)
scoring_population = pd.DataFrame({"ENTITY1": active_ids, "TIMESTAMP": AS_OF})
scoring_population.to_parquet(source_dir / "scoring_population.parquet", index=False)
prediction_table = kumo.PredictionTable.from_source_table(
    files["scoring_population"], pq=query, snapshot=scoring_snapshot, plan=generation_plan)
print("Customers to score:", len(scoring_population))

Customers to score: 2581


### Score the active base

Run the trained model on the new snapshot and save predictions with their scoring time.

In [17]:
prediction_result = trainer.predict(
    prediction_table, snapshot=scoring_snapshot,
    binary_classification_threshold=DECISION_THRESHOLD,
    output_config=PredictionOutputConfig(connector=destination,
        artifact_type=PredictionArtifactType.PREDICTIONS, table_name="predictions.parquet",
        extra_fields=[MetadataField.ANCHOR_TIMESTAMP]),
)
scores = prediction_result.frame

### Inspect the results

There is one row per scored customer: `ENTITY` is the customer ID and `True_PROB` the probability of cancelling within 30 days. Sort by `True_PROB` to build a retention contact list; `TARGET_PRED` applies the untuned `DECISION_THRESHOLD`.

In [18]:
assert len(scores) == len(scoring_population) and set(scores.ENTITY) == set(active_ids)
display(scores.sort_values("True_PROB", ascending=False)[["ENTITY", "ANCHOR_TIMESTAMP", "True_PROB"]].head(10))
print("Saved predictions:", output_dir / "predictions.parquet")

,ENTITY,ANCHOR_TIMESTAMP,True_PROB
1897,2218,2025-07-01,0.103908
890,1911,2025-07-01,0.101261
885,1814,2025-07-01,0.094809
1890,2064,2025-07-01,0.093637
1315,2483,2025-07-01,0.090609
1298,2130,2025-07-01,0.090488
1626,725,2025-07-01,0.090171
255,1139,2025-07-01,0.090048
1519,2586,2025-07-01,0.089504
648,1033,2025-07-01,0.087281


Saved predictions: /home/ubuntu/telco-churn/nb/kumo_example_runs/telco_churn-934wo6cn/output/predictions.parquet


## Retain and reuse

Reload the saved model bundle without retraining. Keep the source tables and graph definition with the bundle: a later run rebuilds the graph and snapshot, then predicts as above. See [publish and retrieve models](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models).

In [19]:
reloaded_trainer = kumo.Trainer.load(output_dir / "model")
print("Reloaded model from:", output_dir / "model")

Reloaded model from: /home/ubuntu/telco-churn/nb/kumo_example_runs/telco_churn-934wo6cn/output/model


## Cleanup

Generated data, the model and predictions stay in `run_dir`. Set `REMOVE_RUN_SCRATCH=True` to delete only this run's scratch directory.

In [20]:
REMOVE_RUN_SCRATCH = False
if REMOVE_RUN_SCRATCH:
    shutil.rmtree(scratch_dir)
    print("Removed run scratch:", scratch_dir)

## Use your own data

Replace the generation cells with your own tables, one folder of Parquet files per table under `source_dir`, and update the `graph` definition to match your columns and keys. The query, training and scoring steps stay the same. Retrain periodically and rescore the active base as each month of data lands.